# 量化精度实验：INT8 / INT4 误差分析 + 可视化

1. 对称/非对称 INT8 量化
2. Per-tensor vs Per-channel
3. INT4 量化
4. SmoothQuant 模拟
5. KV Cache 量化影响

> 面试高频：GPTQ/AWQ/SmoothQuant 原理，INT8 vs INT4 精度权衡

In [ ]:
import numpy as np
try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except Exception:
    HAS_MPL = False
np.random.seed(42)
print("Ready. matplotlib:", HAS_MPL)

## 1) 量化核心函数

In [ ]:
def symmetric_quantize(tensor, bits=8):
    qmax = 2 ** (bits - 1) - 1
    qmin = -(2 ** (bits - 1))
    abs_max = np.max(np.abs(tensor))
    scale = abs_max / qmax if abs_max > 0 else 1.0
    quantized = np.clip(np.round(tensor / scale), qmin, qmax).astype(np.int32)
    dequantized = quantized.astype(np.float32) * scale
    return {"quantized": quantized, "dequantized": dequantized, "scale": scale}

def asymmetric_quantize(tensor, bits=8):
    qmax = 2 ** bits - 1
    x_min, x_max = tensor.min(), tensor.max()
    scale = (x_max - x_min) / qmax if (x_max - x_min) > 0 else 1.0
    zp = int(np.clip(np.round(-x_min / scale), 0, qmax))
    quantized = np.clip(np.round(tensor / scale + zp), 0, qmax).astype(np.int32)
    dequantized = (quantized.astype(np.float32) - zp) * scale
    return {"quantized": quantized, "dequantized": dequantized, "scale": scale, "zero_point": zp}

def per_channel_quantize(weight, bits=8, axis=0):
    qmax = 2 ** (bits - 1) - 1
    qmin = -(2 ** (bits - 1))
    n_ch = weight.shape[axis]
    scales = np.zeros(n_ch)
    dequantized = np.zeros_like(weight, dtype=np.float32)
    quantized = np.zeros_like(weight, dtype=np.int32)
    for c in range(n_ch):
        slc = [slice(None)] * weight.ndim
        slc[axis] = c
        ch = weight[tuple(slc)]
        am = np.max(np.abs(ch))
        s = am / qmax if am > 0 else 1.0
        scales[c] = s
        quantized[tuple(slc)] = np.clip(np.round(ch / s), qmin, qmax).astype(np.int32)
        dequantized[tuple(slc)] = quantized[tuple(slc)].astype(np.float32) * s
    return {"quantized": quantized, "dequantized": dequantized, "scales": scales}

def compute_errors(original, reconstructed):
    diff = original - reconstructed
    mse = np.mean(diff ** 2)
    max_err = np.max(np.abs(diff))
    no, nr = np.linalg.norm(original.flatten()), np.linalg.norm(reconstructed.flatten())
    cos_sim = np.dot(original.flatten(), reconstructed.flatten()) / max(no * nr, 1e-10)
    snr = 10 * np.log10(np.mean(original**2) / max(mse, 1e-10)) if mse > 0 else float('inf')
    return {"mse": float(mse), "max_error": float(max_err), "cosine_sim": float(cos_sim), "snr_db": float(snr)}

w = np.random.randn(256, 512).astype(np.float32)
e8 = compute_errors(w, symmetric_quantize(w, 8)["dequantized"])
e4 = compute_errors(w, symmetric_quantize(w, 4)["dequantized"])
print(f"INT8 sym: MSE={e8['mse']:.6f}, cos={e8['cosine_sim']:.6f}, SNR={e8['snr_db']:.1f}dB")
print(f"INT4 sym: MSE={e4['mse']:.6f}, cos={e4['cosine_sim']:.6f}, SNR={e4['snr_db']:.1f}dB")

## 2) 对比实验：INT8 vs INT4 × Per-tensor vs Per-channel

In [ ]:
weight = np.random.randn(1024, 4096).astype(np.float32) * 0.02
outlier_rows = np.random.choice(1024, 10, replace=False)
weight[outlier_rows, :] *= 5.0  # outlier channels

experiments = [
    ("INT8 per-tensor sym", symmetric_quantize(weight, 8)),
    ("INT4 per-tensor sym", symmetric_quantize(weight, 4)),
    ("INT8 per-tensor asym", asymmetric_quantize(weight, 8)),
    ("INT8 per-channel sym", per_channel_quantize(weight, 8, axis=0)),
    ("INT4 per-channel sym", per_channel_quantize(weight, 4, axis=0)),
]

print(f"{'method':<25} {'MSE':>12} {'MaxErr':>12} {'CosSim':>12} {'SNR(dB)':>10}")
exp_results = []
for name, result in experiments:
    err = compute_errors(weight, result["dequantized"])
    print(f"{name:<25} {err['mse']:>12.8f} {err['max_error']:>12.6f} {err['cosine_sim']:>12.8f} {err['snr_db']:>10.1f}")
    exp_results.append((name, err))

if HAS_MPL:
    names = [r[0] for r in exp_results]
    mses = [r[1]["mse"] for r in exp_results]
    coss = [r[1]["cosine_sim"] for r in exp_results]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    colors = ["tab:blue", "tab:red", "tab:green", "tab:purple", "tab:orange"]
    ax1.barh(range(len(names)), mses, color=colors)
    ax1.set_yticks(range(len(names))); ax1.set_yticklabels(names, fontsize=9)
    ax1.set_xlabel("MSE (lower is better)"); ax1.set_title("Quantization MSE"); ax1.grid(axis="x", alpha=0.3)
    ax2.barh(range(len(names)), coss, color=colors)
    ax2.set_yticks(range(len(names))); ax2.set_yticklabels(names, fontsize=9)
    ax2.set_xlabel("Cosine Similarity"); ax2.set_title("Cosine Similarity"); ax2.set_xlim(min(coss)*0.999, 1.0001)
    ax2.grid(axis="x", alpha=0.3)
    plt.tight_layout(); plt.show()

## 3) SmoothQuant 模拟：解决 Activation Outlier

In [ ]:
activation = np.random.randn(32, 4096).astype(np.float32)
for ch in [100, 500, 2000]:
    activation[:, ch] *= 20.0

def smooth_quant(activation, weight, alpha=0.5):
    # activation: (B, hidden_dim), weight: (out_dim, hidden_dim)
    # s_j per hidden_dim channel
    act_max = np.maximum(np.max(np.abs(activation), axis=0), 1e-5)  # (hidden_dim,)
    w_max = np.maximum(np.max(np.abs(weight), axis=0), 1e-5)       # (hidden_dim,)
    s = np.power(act_max, alpha) / np.power(w_max, 1.0 - alpha)
    return activation / s[None, :], weight * s[None, :], s

w_sm = np.random.randn(1024, 4096).astype(np.float32) * 0.02  # (out_dim, hidden_dim)
s_act, s_weight, _ = smooth_quant(activation, w_sm, alpha=0.5)

err_orig = compute_errors(activation, symmetric_quantize(activation, 8)["dequantized"])
err_smooth = compute_errors(s_act, symmetric_quantize(s_act, 8)["dequantized"])
print("Activation INT8 quantization:")
print(f"  Original:  MSE={err_orig['mse']:.6f}, CosSim={err_orig['cosine_sim']:.6f}")
print(f"  Smoothed:  MSE={err_smooth['mse']:.6f}, CosSim={err_smooth['cosine_sim']:.6f}")
print(f"  MSE reduction: {err_orig['mse'] / max(err_smooth['mse'], 1e-10):.1f}x")

## 4) GEMM 量化误差 & KV Cache 量化

In [ ]:
X = np.random.randn(32, 4096).astype(np.float32)
W = np.random.randn(1024, 4096).astype(np.float32) * 0.02
Y_ref = X @ W.T

W_q8 = symmetric_quantize(W, 8)["dequantized"]
W_q4 = symmetric_quantize(W, 4)["dequantized"]
X_q8 = symmetric_quantize(X, 8)["dequantized"]

gemm_results = []
for name, Y_hat in [("W8A16", X @ W_q8.T), ("W4A16", X @ W_q4.T), ("W8A8", X_q8 @ W_q8.T)]:
    err = compute_errors(Y_ref, Y_hat)
    print(f"{name:<8} MSE={err['mse']:.6f} CosSim={err['cosine_sim']:.6f}")
    gemm_results.append((name, Y_hat))

if HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (name, Y_hat) in zip(axes, gemm_results):
        err_map = np.abs(Y_ref - Y_hat)[:16, :64]
        im = ax.imshow(err_map, aspect="auto", cmap="hot")
        ax.set_title(f"{name} |error|"); plt.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout(); plt.show()

In [ ]:
# KV Cache 量化
batch, n_heads, seq_len, head_dim = 4, 32, 2048, 128
K = np.random.randn(batch, n_heads, seq_len, head_dim).astype(np.float32) * 0.5
V = np.random.randn(batch, n_heads, seq_len, head_dim).astype(np.float32) * 0.5
Q = np.random.randn(batch, n_heads, 1, head_dim).astype(np.float32) * 0.5

def simple_attention(Q, K, V):
    d = Q.shape[-1]
    scores = Q @ K.transpose(0, 1, 3, 2) / np.sqrt(d)
    w = np.exp(scores - scores.max(axis=-1, keepdims=True))
    w = w / w.sum(axis=-1, keepdims=True)
    return w @ V

Y_fp32 = simple_attention(Q, K, V)
Ks, Vs = K.shape, V.shape
K8 = symmetric_quantize(K.reshape(-1), 8)["dequantized"].reshape(Ks)
V8 = symmetric_quantize(V.reshape(-1), 8)["dequantized"].reshape(Vs)
K4 = symmetric_quantize(K.reshape(-1), 4)["dequantized"].reshape(Ks)
V4 = symmetric_quantize(V.reshape(-1), 4)["dequantized"].reshape(Vs)

ekv8 = compute_errors(Y_fp32, simple_attention(Q, K8, V8))
ekv4 = compute_errors(Y_fp32, simple_attention(Q, K4, V4))
kv_bytes = 2 * batch * n_heads * seq_len * head_dim
print(f"KV Cache Quantization Impact:")
print(f"  INT8 KV: MSE={ekv8['mse']:.8f}, CosSim={ekv8['cosine_sim']:.8f}")
print(f"  INT4 KV: MSE={ekv4['mse']:.8f}, CosSim={ekv4['cosine_sim']:.8f}")
print(f"Memory: FP16={kv_bytes*2/1024/1024:.1f}MB  INT8={kv_bytes/1024/1024:.1f}MB  INT4={kv_bytes//2/1024/1024:.1f}MB")

## 5) 面试讲解模板

| 方案 | 量化对象 | 精度损失 | 显存节省 | 适用场景 |
|------|---------|---------|---------|--------|
| W8A16 | 仅权重 INT8 | 极小 | 权重减半 | 通用 |
| W4A16 (GPTQ/AWQ) | 仅权重 INT4 | 较小 | 权重 4x 压缩 | 显存受限 |
| W8A8 (SmoothQuant) | 权重+激活 INT8 | 小 | 两者都减半 | 高吞吐 |
| KV INT8 | KV Cache INT8 | 极小 | KV 减半 | 长上下文 |

### 面试追问
1. **GPTQ vs AWQ？** → GPTQ: layer-wise OBQ; AWQ: 保护显著 channel
2. **FP8？** → E4M3/E5M2, H100 原生支持
3. **量化后 GEMM？** → 整数乘加 + FP32 累加 + scale
4. **何时不量化？** → 模型很小 / 精度极高要求 / 训练阶段